# 🛠️ Ollama & Hugging Face Model-Download Utility
**Projekt: Mai_AI (MaiOmni) — Dynamisches Dienstprogramm zur Modellverwaltung**

Dieses Notebook dient als **interaktives Cockpit mit dynamischem Layout-Management** für das Abfragen, Suchen/Filtern, Herunterladen und automatische Einbinden von Offline-Modellen in deine lokale **Ollama-Laufzeitumgebung**.

---
### 🎯 Die 3 klar getrennten Sektionen:
1. **Sektion 1 (Plattform-Wahl):** Auswahl der Modellquelle (**Ollama Library** oder **Hugging Face GGUF**) mit Bestätigungs-Button.
2. **Sektion 2 (Echtzeit-Suche & Modellauswahl):** Dynamischer Live-Filter für Modelle sowie visuelle Eignungs-Symbole (✅ / ❌) für GGUF-Varianten passend zur erkannten Hardware.
3. **Sektion 3 (Download & Einbindung):** Autonomer Hintergrund-Download mit Live-Fortschrittsbalken und automatischer Registrierung in Ollama.

---

### ⚙️ Schritt 1: System- & Dienst-Initialisierung
Überprüfung aller erforderlichen Python-Abhängigkeiten und plattformübergreifender Health-Check des lokalen Ollama-Dienstes.

In [12]:
# Absicherung der Konsolenausgabe gegen Windows-Encoding-Fehler (cp1252)
import sys
if sys.stdout and hasattr(sys.stdout, "reconfigure"):
    try:
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    except Exception:
        pass

import os
import json
import subprocess
import importlib
import platform
import time
import shutil
import urllib.request
import psutil

# 1. Selbstheilende Abhängigkeitsprüfung
required_packages = {
    "ipywidgets": "ipywidgets",
    "ollama": "ollama",
    "psutil": "psutil",
    "huggingface_hub": "huggingface_hub",
    "tqdm": "tqdm"
}

missing_packages = []
for module_name, package_name in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        missing_packages.append(package_name)

if missing_packages:
    print(f"[AUTO-REPARATUR] Installiere fehlende Pakete: {missing_packages}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
    print("[✓] Abhängigkeiten erfolgreich bereitgestellt.")

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import ollama
from ollama import Client
from huggingface_hub import HfApi, hf_hub_download

# 2. Projektpfade ermitteln
from pathlib import Path

def get_project_root() -> str:
    """Ermittelt die Projektwurzel robust über pathlib, unabhängig von OS oder Laufwerksbuchstaben."""
    try:
        if "__file__" in globals():
            current_path = Path(__file__).resolve()
        else:
            current_path = Path.cwd().resolve()
        
        # Durchsuche den Pfad nach oben, bis der Projektordner "offline_ai" gefunden wird
        for parent in [current_path] + list(current_path.parents):
            if parent.name.lower() == "offline_ai":
                return str(parent)
                
    except Exception as e:
        # Fehler gezielt ausgeben statt ihn zu verschlucken
        print(f"[FEHLER] get_project_root() konnte Projektwurzel nicht ermitteln: {e}")
        
    # Fallback mit Benachrichtigung
    fallback_path = Path.cwd().resolve()
    print(f"[INFO] get_project_root() nutzt Fallback: {fallback_path}")
    return str(fallback_path)

PROJECT_ROOT = get_project_root()
CONFIG_DIR = os.path.join(PROJECT_ROOT, "config")
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
HF_MODELS_DIR = os.path.join(DATA_DIR, "models", "huggingface")
os.makedirs(CONFIG_DIR, exist_ok=True)
os.makedirs(HF_MODELS_DIR, exist_ok=True)

# 3. Hardware-Spezifikationen ermitteln
TOTAL_RAM_GB = round(psutil.virtual_memory().total / (1024**3), 2)
SAFE_RAM_BUDGET_GB = round(TOTAL_RAM_GB * 0.70, 1)
CPU_COUNT = psutil.cpu_count(logical=True)

# 4. Plattformunabhängiger Ollama-Service-Manager
class OllamaServiceManager:
    def __init__(self, host="127.0.0.1", port=11434):
        self.host = host
        self.port = port
        self.client = Client(host=f"http://{host}:{port}")

    def is_running(self, timeout=1.5) -> bool:
        try:
            Client(host=f"http://{self.host}:{self.port}", timeout=timeout).list()
            return True
        except Exception:
            # Hier kein Print bei jedem Loop-Tick, da das Warten anfangs normal ist,
            # aber der Timeout am Ende gibt Aufschluss.
            return False

    def _is_process_running(self, process_name="ollama") -> bool:
        """Prüft systemübergreifend, ob ein Prozess mit diesem Namen bereits im Hintergrund läuft."""
        try:
            for proc in psutil.process_iter(['name']):
                if proc.info['name'] and process_name.lower() in proc.info['name'].lower():
                    return True
        except Exception as e:
            print(f"[FEHLER] _is_process_running() konnte Prozessliste nicht abrufen: {e}")
        return False

    def ensure_service(self, max_wait_seconds=20) -> bool:
        if self.is_running():
            return True
        
        # Verhindere das Erzeugen von doppelten Prozessen
        if self._is_process_running("ollama"):
            print("[INFO] Ollama-Prozess läuft bereits im System, warte auf Bereitschaft...")
        else:
            print("[WARNUNG] Ollama-Dienst nicht aktiv. Starte Hintergrundprozess...")
            current_os = platform.system()
            try:
                if current_os == "Windows":
                    candidates = ["ollama.exe", os.path.expanduser("~\\AppData\\Local\\Programs\\Ollama\\ollama.exe")]
                    found_candidate = False
                    for cand in candidates:
                        found = shutil.which(cand) or (cand if os.path.exists(cand) else None)
                        if found:
                            found_candidate = True
                            creationflags = subprocess.CREATE_NEW_PROCESS_GROUP if hasattr(subprocess, 'CREATE_NEW_PROCESS_GROUP') else 0
                            subprocess.Popen([found, "serve"], creationflags=creationflags, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                            break
                    if not found_candidate:
                        print(f"[FEHLER] Keine Ollama-Executable unter den Kandidaten gefunden: {candidates}")
                elif current_os == "Darwin":
                    try:
                        subprocess.Popen(["open", "-a", "Ollama"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                    except Exception as e:
                        print(f"[INFO] 'open -a Ollama' fehlgeschlagen ({e}), versuche direktes Starten...")
                        subprocess.Popen(["ollama", "serve"], start_new_session=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                else:
                    try:
                        subprocess.run(["systemctl", "start", "ollama"], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                    except Exception as e:
                        print(f"[INFO] 'systemctl start ollama' fehlgeschlagen ({e}), versuche direktes Starten...")
                        subprocess.Popen(["ollama", "serve"], start_new_session=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            except Exception as e:
                print(f"[FEHLER] Beim Startversuch des Ollama-Dienstes ist ein Fehler aufgetreten: {e}")
        
        start_time = time.time()
        while time.time() - start_time < max_wait_seconds:
            time.sleep(1)
            if self.is_running():
                print("[✓] Ollama-Dienst erfolgreich verbunden!")
                return True
                
        print(f"[FEHLER] Timeout: Ollama-Dienst ist innerhalb von {max_wait_seconds} Sekunden nicht erreichbar geworden.")
        return False


service_manager = OllamaServiceManager()
ollama_ready = service_manager.ensure_service()

print("=" * 65)
print(f"[✓] SYSTEM-STATUS: {platform.system()} ({platform.machine()})")
print(f"    -> CPU: {CPU_COUNT} Kerne | RAM: {TOTAL_RAM_GB} GB (Budget: {SAFE_RAM_BUDGET_GB} GB)")
print(f"    -> Ollama-Dienst: {'[✓] Verbunden' if ollama_ready else '[!] Nicht erreichbar'}")
print("=" * 65)

[✓] SYSTEM-STATUS: Windows (AMD64)
    -> CPU: 8 Kerne | RAM: 15.85 GB (Budget: 11.1 GB)
    -> Ollama-Dienst: [✓] Verbunden


### 🔌 Schritt 2: Backend-Logik & Dynamische API-Schnittstellen
Trennung der Datenbeschaffung von der Benutzeroberfläche: Die Klassen `OllamaBackend` und `HuggingFaceBackend` rufen Modelle live ohne Hardcoding ab und bewerten automatisch die Hardware-Eignung.

In [ ]:
# BACKEND: OLLAMA API & LIVE LIBRARY FETCHER

import os
import re
import threading
from pathlib import Path

_DOWNLOAD_LOCK = threading.Lock()

OLLAMA_TAG_PATTERN = re.compile(
    r"^[a-z0-9]+[a-z0-9._/-]*(?::[a-z0-9][a-z0-9._-]*)?$"
)

def normalize_model_tag(value: str) -> str:
    """
    Normalisiert und validiert einen Ollama Model-Tag.
    Wirft ValueError bei ungültigen Tags.
    """
    tag = str(value or "").strip().lower()
    print(, [DEBUG] Validiere Model-Tag: {tag!r}")

    if not tag:
        print("[FEHLER] Model-Tag ist leer.")
        raise ValueError("Model tag must not be empty")

    if len(tag) > 200:
        print(f"[FEHLER] Model-Tag überschreitet die maximale Länge von 200 Zeichen (Länge: {len(tag)}).")
        raise ValueError("Model tag is too long (max 200 characters)")

    if not OLLAMA_TAG_PATTERN.fullmatch(tag):
        print(f"[FEHLER] Model-Tag entspricht nicht dem erlaubten Muster: {tag!r}")
        raise ValueError(
            f"Invalid Ollama model tag: {tag!r}. "
            "Use lowercase letters, digits, '.', '_', '/', '-' and optional ':' tag."
        )

    print(f"[✓] Model-Tag erfolgreich normalisiert: {tag}")
    return tag

import urllib.request
import json
from ollama import Client

class OllamaBackend:
    ONLINE_LIBRARY_URL = "https://raw.githubusercontent.com/chrizzo84/OllamaScraper/refs/heads/main/out/ollama_models.json"

    def __init__(self, host="127.0.0.1", port=11434):
        self.host = host
        self.port = port
        self.client = Client(host=f"http://{host}:{port}")

    def fetch_models(self, limit=80) -> list[dict]:
        """Ruft die verfügbaren Modelle dynamisch aus der offiziellen Library ab (Fallback: lokaler Daemon)."""
        models = []
        print(f"[INFO] Versuche, Modelle von Online-Bibliothek abzurufen: {self.ONLINE_LIBRARY_URL}")
        
        try:
            req = urllib.request.Request(self.ONLINE_LIBRARY_URL, headers={"User-Agent": "Offline_AI/1.0"})
            with urllib.request.urlopen(req, timeout=4.0) as response:
                data = json.loads(response.read().decode("utf-8"))
                raw_models = data.get("models", []) if isinstance(data, dict) else (data if isinstance(data, list) else [])
                print(f"[INFO] {len(raw_models)} Roheinträge aus Online-Bibliothek geladen.")
                
                for item in raw_models[:limit]:
                    name = item.get("name", "")
                    pulls = item.get("pulls_text", "")
                    capabilities = item.get("capabilities", [])
                    variants = item.get("variants", [])
                    blurb = item.get("blurb", "") or item.get("description", "")
                    category = ", ".join(capabilities) if capabilities else "Allgemein / LLM"
                    
                    if variants:
                        for var in variants[:4]:
                            tag = var.get("tag", name)
                            size_text = var.get("size_text", "Variabel")
                            models.append({
                                "id": tag,
                                "name": tag,
                                "size": size_text,
                                "category": category,
                                "description": blurb,
                                "popularity": pulls,
                                "platform": "ollama"
                            })
                    else:
                        models.append({
                            "id": name,
                            "name": name,
                            "size": "Variabel",
                            "category": category,
                            "description": blurb,
                            "popularity": pulls,
                            "platform": "ollama"
                        })
                if models:
                    print(f"[✓] {len(models)} Modelle erfolgreich aus Online-Bibliothek verarbeitet.")
                    return models
                    
        except Exception as err:
            print(f"[HINWEIS] Online-Bibliothek nicht erreichbar oder fehlerhaft ({err}). Nutze lokalen Ollama-Daemon als Fallback...")

        # Fallback: Lokale Tags über API
        print(f"[INFO] Frage lokale Modelle über Ollama-Client ab (Host: {self.host}:{self.port})...")
        try:
            local_tags = self.client.list()
            for m in local_tags.models:
                size_mb = m.size // (1024**2) if m.size else 0
                family = getattr(m.details, "family", "lokal") or "lokal"
                models.append({
                    "id": m.model,
                    "name": m.model,
                    "size": f"{size_mb} MB",
                    "category": family,
                    "description": f"Lokal installiertes Modell ({family})",
                    "popularity": "Lokal vorhanden",
                    "platform": "ollama"
                })
            print(f"[✓] {len(models)} lokale Modelle erfolgreich über den Daemon geladen.")
        except Exception as err:
            print(f"[FEHLER] Lokale Modelle konnten nicht über den Client abgefragt werden: {err}")

        return models

    def pull_model(self, model_tag: str, progress_callback=None):
        """Führt den Streaming-Pull eines Modells über den Ollama Client aus mit Fehler-Logging."""
        print(f"[INFO] Starte Download (Pull) für Modell: {model_tag!r}")
        try:
            for progress in self.client.pull(model=model_tag, stream=True):
                if progress_callback:
                    progress_callback(progress)
            print(f"[✓] Modell-Pull erfolgreich abgeschlossen: {model_tag!r}")
        except Exception as e:
            print(f"[FEHLER] Beim Download (Pull) des Modells {model_tag!r} ist ein Fehler aufgetreten: {e}")
            raise

# BACKEND: HUGGING FACE API & GGUF INTEGRATION MIT HARDWARE-FIT CHECK
class HuggingFaceBackend:
    OLLAMA_TAG_PATTERN = re.compile(r"^[a-z0-9]+[a-z0-9._/-]*(?::[a-z0-9][a-z0-9._-]*)?$")

    @staticmethod
    def normalize_model_tag(value: str) -> str:
        tag = str(value or "").strip().lower()
        print(f"[DEBUG] Validiere HuggingFace/Ollama Model-Tag: {tag!r}")

        if not tag:
            print("[FEHLER] Model-Tag ist leer.")
            raise ValueError("Model tag must not be empty")

        if len(tag) > 200:
            print(f"[FEHLER] Model-Tag überschreitet die maximale Länge von 200 Zeichen (Länge: {len(tag)}).")
            raise ValueError("Model tag is too long (max 200 characters)")

        if not HuggingFaceBackend.OLLAMA_TAG_PATTERN.fullmatch(tag):
            print(f"[FEHLER] Model-Tag entspricht nicht dem erlaubten Muster: {tag!r}")
            raise ValueError(
                f"Invalid Ollama model tag: {tag!r}. "
                "Use lowercase letters, digits, '.', '_', '/', '-' and optional ':' tag."
            )

        print(f"[✓] Model-Tag erfolgreich validiert: {tag}")
        return tag

    def __init__(self):
        self.api = HfApi()

    def fetch_models(self, limit=50) -> list[dict]:
        """Ruft die beliebtesten GGUF-Modelle dynamisch von Hugging Face ab."""
        models = []
        print(f"[INFO] Starte Abruf der Top-{limit} GGUF-Modelle von Hugging Face...")
        try:
            results = self.api.list_models(
                filter="gguf",
                sort="downloads",
                limit=limit
            )
            for m in results:
                downloads = getattr(m, "downloads", 0)
                pipeline = getattr(m, "pipeline_tag", "text-generation") or "text-generation"
                models.append({
                    "id": m.id,
                    "name": m.id,
                    "size": "GGUF Repository",
                    "category": pipeline,
                    "description": f"Hugging Face GGUF Repository: {m.id} | Task: {pipeline}",
                    "popularity": f"{downloads:,} Pull Downloads",
                    "platform": "huggingface"
                })
            print(f"[✓] {len(models)} GGUF-Modelle erfolgreich von Hugging Face geladen.")
        except Exception as err:
            print(f"[FEHLER] Hugging Face API-Abruf fehlgeschlagen: {err}")
        return models

    def fetch_gguf_files_with_fit(self, repo_id: str) -> list[tuple[str, str]]:
        """
        Ruft alle .gguf-Dateien eines Repositories dynamisch ab und markiert sie 
        mit einem visuellen Eignungs-Symbol.
        """
        options = []
        print(f"[INFO] Lade Dateibaum für Repository: {repo_id!r}")
        try:
            tree_items = list(self.api.list_repo_tree(repo_id=repo_id))
            gguf_items = [item for item in tree_items if item.path.lower().endswith(".gguf")]
            print(f"[INFO] {len(gguf_items)} .gguf-Dateien im Repository-Baum gefunden.")
            
            # Sortierung: bevorzugte Quantisierungen zuerst
            sorted_items = sorted(gguf_items, key=lambda x: (not ("q4_k_m" in x.path.lower() or "q4_0" in x.path.lower()), x.path))
            
            for item in sorted_items:
                path = item.path
                size_gb = round(item.size / (1024**3), 2) if hasattr(item, "size") and item.size else None
                
                if size_gb is not None:
                    if size_gb <= SAFE_RAM_BUDGET_GB:
                        label = f"✅ {path} ({size_gb} GB - Passt zur Hardware)"
                    else:
                        label = f"❌ {path} ({size_gb} GB - Übersteigt RAM-Budget von {SAFE_RAM_BUDGET_GB} GB)"
                else:
                    label = f"ℹ️ {path}"
                
                options.append((label, path))
                
        except Exception as err:
            print(f"[HINWEIS] Tree-Abruf für '{repo_id}' fehlgeschlagen ({err}). Nutze Fallback-Dateiliste...")
            try:
                files = [f for f in self.api.list_repo_files(repo_id=repo_id) if f.lower().endswith(".gguf")]
                print(f"[INFO] Fallback: {len(files)} .gguf-Dateien über Dateiliste gefunden.")
                for f in files:
                    options.append((f"ℹ️ {f}", f))
            except Exception as f_err:
                print(f"[FEHLER] Dateiliste konnte über Fallback ebenfalls nicht geladen werden: {f_err}")
                
        return options

    def download_and_create_model(
        self,
        repo_id: str,
        filename: str,
        target_model_tag: str,
        dest_dir: str,
        progress_callback=None,
        build_callback=None,
    ):
        """
        Läd eine GGUF-Datei herunter und registriert das Modell in Ollama, 
        falls es nicht bereits vorhanden ist.
        """
        filename = Path(filename).name.strip()
        print(f"[INFO] Starte Download-Prozess für '{filename}' aus Repository '{repo_id}'...")

        if not filename or not filename.lower().endswith(".gguf"):
            print(f"[FEHLER] Ungültiger Dateiname übergeben: {filename!r}")
            raise ValueError("filename must be a valid .gguf filename")

        clean_model_tag = self.normalize_model_tag(target_model_tag)

        destination = Path(dest_dir).resolve()
        destination.mkdir(parents=True, exist_ok=True)

        local_path = destination / filename
        temporary_path = local_path.with_suffix(local_path.suffix + ".part")
        ollama_client = client_module.Client(host="http://127.0.0.1:11434") if 'client_module' in globals() else ollama.Client(host="http://127.0.0.1:11434")

        with _DOWNLOAD_LOCK:
            # 1. Vorhandene Datei wiederverwenden
            if local_path.is_file() and local_path.stat().st_size > 0:
                msg = f"Lokale GGUF-Datei '{filename}' ist bereits vorhanden. Verwende vorhandene Datei."
                print(f"[INFO] {msg}")
                if progress_callback:
                    progress_callback(msg)
            else:
                if temporary_path.exists():
                    print(f"[INFO] Lösche unvollständige temporäre Datei: {temporary_path}")
                    temporary_path.unlink()

                msg = f"Lade '{filename}' von Hugging Face ({repo_id}) herunter..."
                print(f"[INFO] {msg}")
                if progress_callback:
                    progress_callback(msg)

                downloaded_path = hf_hub_download(
                    repo_id=repo_id,
                    filename=filename,
                    local_dir=str(destination),
                    local_dir_use_symlinks=False,
                )

                downloaded_path = Path(downloaded_path).resolve()

                if not downloaded_path.is_file() or downloaded_path.stat().st_size <= 0:
                    print(f"[FEHLER] Heruntergeladene Datei ist leer oder ungültig: {downloaded_path}")
                    raise IOError(
                        f"Download fehlgeschlagen oder Datei ist leer: {downloaded_path}"
                    )

                if downloaded_path != local_path:
                    os.replace(str(downloaded_path), str(local_path))

        abs_local_path = str(local_path.resolve())
        file_size_gb = round(local_path.stat().st_size / (1024**3), 2)
        print(f"[✓] Datei erfolgreich bereitgestellt unter {abs_local_path} ({file_size_gb} GB)")

        # 2. Existenzprüfung auf dem Client
        try:
            existing_models = ollama_client.list().models
            existing_names = {
                str(model.model).strip().lower()
                for model in existing_models
            }
        except Exception as err:
            print(f"[FEHLER] Konnte installierte Ollama-Modelle für den Abgleich nicht abfragen: {err}")
            raise RuntimeError(
                f"Ollama-Modelle konnten nicht abgefragt werden: {err}"
            ) from err

        if clean_model_tag in existing_names:
            msg = f"Modell '{clean_model_tag}' ist bereits in Ollama registriert. Überspringe Registrierung."
            print(f"[INFO] {msg}")
            if progress_callback:
                progress_callback(msg)
            return abs_local_path, file_size_gb

        # 3. Modellregistrierung triggern
        msg = f"Registriere Modell '{clean_model_tag}' in Ollama..."
        print(f"[INFO] {msg}")
        if progress_callback:
            progress_callback(msg)

        try:
            for response in ollama_client.create(
                model=clean_model_tag,
                from_=abs_local_path,
                stream=True,
            ):
                status = response.get("status", "")
                if status:
                    print(f"[OLLAMA CREATE] {status}")
                if build_callback:
                    build_callback(status)
            print(f"[✓] Modell '{clean_model_tag}' erfolgreich registriert!")
        except Exception as e:
            print(f"[FEHLER] Fehler während der Ollama-Modellregistrierung: {e}")
            raise

        return abs_local_path, file_size_gb

# CONFIG MANAGER
class ModelConfigManager:
    @staticmethod
    def save_active_model(model_name: str, size_gb: float, platform_name: str, config_dir: str):
        print(f"[INFO] Speichere aktive Modellkonfiguration für '{model_name}' (Plattform: {platform_name})...")
        try:
            config_path = os.path.join(config_dir, "active_model_config.json")
            os.makedirs(config_dir, exist_ok=True)
            data = {
                "model_name": model_name,
                "allocated_size_gb": size_gb,
                "detected_ram_gb": TOTAL_RAM_GB,
                "platform": platform_name,
                "updated_at": time.strftime('%Y-%m-%d %H:%M:%S')
            }
            with open(config_path, "w", encoding="utf-8") as f:
                json.dump(data, f, indent=4, ensure_ascii=False)
            print(f"[✓] Modellkonfiguration erfolgreich unter '{config_path}' gespeichert.")
        except Exception as e:
            print(f"[FEHLER] Konnte aktive Modellkonfiguration nicht speichern: {e}")
            raise

ollama_backend = OllamaBackend()
hf_backend = HuggingFaceBackend()
print("[✓] Backend-Module erfolgreich initialisiert.")

[✓] Backend-Module erfolgreich initialisiert.


### 🎛️ Schritt 3: Interaktive Benutzeroberfläche mit dynamischem Layout-Management & Hardware-Symbolen
Führe die Zelle aus, um das strukturierte UI-Cockpit zu starten. Jede Sektion ist visuell voneinander isoliert, und GGUF-Varianten werden mit Eignungssymbolen (**✅ / ❌**) gekennzeichnet.

In [ ]:
# --- DYNAMISCHES LAYOUT-MANAGEMENT & WIDGET-INITIALISIERUNG ---
import threading
import os
import json
import psutil
import time
import subprocess
import ollama
from pathlib import Path

LABEL_WIDTH = "120px"
FULL_WIDTH = "100%"
MAX_WIDTH = "760px"

# --- PERSISTENTE PFAD-SPEICHERUNG LOGIK ---
PATH_CONFIG_FILE = os.path.join(CONFIG_DIR, "last_storage_path.json")

def load_last_storage_path() -> str:
    """Lädt den zuletzt verwendeten Speicherpfad aus der Konfiguration."""
    print(f"[INFO] Versuche, letzten Speicherpfad aus '{PATH_CONFIG_FILE}' zu laden...")
    try:
        if os.path.exists(PATH_CONFIG_FILE):
            with open(PATH_CONFIG_FILE, "r", encoding="utf-8") as f:
                data = json.load(f)
                path = data.get("path", "")
                if path:
                    print(f"[✓] Speicherpfad erfolgreich geladen: {path!r}")
                else:
                    print(f"[INFO] Konfigurationsdatei existiert, enthält aber keinen 'path'-Eintrag.")
                return path
        else:
            print(f"[INFO] Konfigurationsdatei '{PATH_CONFIG_FILE}' existiert noch nicht. Nutze Fallback (Leerpfad).")
    except Exception as e:
        print(f"[FEHLER] Konnte letzten Speicherpfad nicht aus '{PATH_CONFIG_FILE}' laden: {e}")
    return ""

def save_last_storage_path(path: str):
    """Speichert den gewählten Pfad persistent ab mit aktivem Logging."""
    print(f"[INFO] Speichere letzten Speicherpfad: {path!r} in '{PATH_CONFIG_FILE}'...")
    try:
        os.makedirs(CONFIG_DIR, exist_ok=True)
        with open(PATH_CONFIG_FILE, "w", encoding="utf-8") as f:
            json.dump({"path": path}, f, indent=4, ensure_ascii=False)
        print(f"[✓] Speicherpfad erfolgreich unter '{PATH_CONFIG_FILE}' gespeichert.")
    except Exception as e:
        print(f"[FEHLER] Konnte letzten Speicherpfad nicht speichern: {e}")
        raise

# --- LAUFWERKS- & PFAD-ERKENNUNG ---
def get_system_drives() -> list[tuple[str, str]]:
    """Ermittelt alle verfügbaren Laufwerke/Mountpoints inklusive freiem Speicher mit aktivem Logging."""
    drive_options = []
    print("[INFO] Ermittle verfügbare Systemlaufwerke und Speicherplätze...")
    try:
        for part in psutil.disk_partitions(all=False):
            try:
                usage = psutil.disk_usage(part.mountpoint)
                free_gb = round(usage.free / (1024**3), 1)
                total_gb = round(usage.total / (1024**3), 1)
                label = f"💽 {part.device} ({part.mountpoint}) - {free_gb} GB frei (von {total_gb} GB)"
                target_path = os.path.join(part.mountpoint, "Offline_AI_Models")
                drive_options.append((label, target_path))
                print(f"[DEBUG] Laufwerk erfolgreich eingelesen: {part.device} ({part.mountpoint}) - {free_gb} GB frei")
            except Exception as e:
                print(f"[HINWEIS] Konnte Festplattendaten für Mountpoint '{getattr(part, 'mountpoint', 'unbekannt')}' nicht abrufen: {e}")
    except Exception as e:
        print(f"[FEHLER] Fehler beim Abrufen der Festplattenpartitionen über psutil: {e}")
    if not drive_options:
        default_path = os.path.join(DATA_DIR, "models")
        print(f"[WARNUNG] Keine Laufwerke gefunden. Verwende Standard-Fallback-Pfad: {default_path}")
        drive_options.append((f"Standard ({default_path})", default_path))
    drive_options.append(("➕ Benutzerdefinierten Pfad eingeben...", "CUSTOM_PATH"))
    print(f"[✓] Laufwerksermittlung abgeschlossen. {len(drive_options)} Optionen bereitgestellt.")
    return drive_options

# --- UI CONTROLLER & EVENT LOGIC ---
CURRENT_LOADED_MODELS = []

def filter_model_options(query: str = ""):
    """Filtert die geladenen Modelle in Echtzeit anhand des Suchbegriffs und sortiert sie alphabetisch (A-Z) mit aktivem Logging."""
    platform_type = dropdown_platform.value
    q = query.strip().lower()
    print(f"[INFO] Starte Modellfilterung für Plattform '{platform_type}' mit Suchbegriff: {q!r}")
    
    if not CURRENT_LOADED_MODELS:
        print("[WARNUNG] Keine Modelle in 'CURRENT_LOADED_MODELS' vorhanden. Setze Dropdown auf 'Keine Modelle geladen'.")
        dropdown_model.options = [("Keine Modelle geladen", "")]
        dropdown_model.disabled = True
        btn_start_download.disabled = True
        return
    
    print(f"[INFO] Gesamtzahl verfügbarer Modelle vor Filterung: {len(CURRENT_LOADED_MODELS)}")
    matched_models = []
    for m in CURRENT_LOADED_MODELS:
        searchable_content = f"{m.get('name', '')} {m.get('id', '')} {m.get('category', '')} {m.get('size', '')} {m.get('description', '')} {m.get('popularity', '')}".lower()
        if not q or q in searchable_content:
            matched_models.append(m)
            
    # --- LOGIK: ALPHABETISCHE SORTIERUNG NACH NAME (CASE-INSENSITIVE) ---
    matched_models = sorted(matched_models, key=lambda x: str(x.get('name', '')).lower())
    print(f"[INFO] {len(matched_models)} Modelle nach Filterung und alphabetischer Sortierung übrig.")
    
    if platform_type == "ollama":
        options = [(f"{m['name']} ({m['size']}) [{m['category']}]", m['id']) for m in matched_models]
        options.append(("Benutzerdefiniertes Modell (Tag eingeben)", "CUSTOM_OLLAMA"))
    else:
        options = [(f"{m['name']} [{m['category']}] ({m['popularity']})", m['id']) for m in matched_models]
        options.append(("Benutzerdefiniertes Hugging Face Modell (Repo eingeben)", "CUSTOM_HF"))
    
    if matched_models or q == "":
        dropdown_model.options = options
        dropdown_model.disabled = False
        btn_start_download.disabled = False
        
        # --- VERBESSERUNG & SICHERHEITS-CHECK FÜR DIE VORAUSWAHL ---
        if len(options) > 0:
            first_selected_id = options[0][1]
            dropdown_model.value = first_selected_id
            print(f"[✓] Automatische Vorauswahl gesetzt auf ID: {first_selected_id!r}")
            
            # Schaltet sofort das GGUF-Feld scharf und lädt die Quantisierungen
            if platform_type == "huggingface" and first_selected_id != "CUSTOM_HF":
                print(f"[INFO] Lade Hugging Face Dateidownload-Optionen für Vorauswahl: {first_selected_id!r}")
                update_hf_file_dropdown(first_selected_id)
                
            matched_item = next((m for m in CURRENT_LOADED_MODELS if m['id'] == first_selected_id), None)
            if matched_item:
                html_spec_card.value = render_spec_card(matched_item)
    else:
        print(f"[INFO] Keine Treffer für Suchbegriff '{query}' gefunden.")
        dropdown_model.options = [(f"Keine Treffer für '{query}'", "")]
        dropdown_model.disabled = True
        btn_start_download.disabled = True
        html_spec_card.value = "<div style='padding: 8px 10px; color: #DC2626; font-size: 12px;'>Keine passenden Modelle gefunden. Bitte Suchbegriff anpassen.</div>"

# --- DOWNLOAD EXECUTION MIT VERTEILTER PFAD-ERZWINGUNG ---
def run_download_process():
    """Führt den vollständigen Download- und Integrationsprozess mit umfassendem Logging aus."""
    platform_choice = dropdown_platform.value
    selected_id = dropdown_model.value
    
    print(f"[INFO] Starte Download-Prozess für Plattform: {platform_choice!r}, Modell-ID: {selected_id!r}")
    
    if dropdown_target_disk.value == "CUSTOM_PATH":
        custom_path_val = text_custom_storage_path.value.strip()
        if not custom_path_val:
            print("[FEHLER] Kein gültiger benutzerdefinierter Speicherpfad angegeben.")
            raise ValueError("Bitte einen gültigen benutzerdefinierten Speicherpfad angeben!")
        active_dest_dir = custom_path_val
    else:
        active_dest_dir = dropdown_target_disk.value

    print(f"[INFO] Aktives Zielverzeichnis festgelegt auf: {active_dest_dir}")
    save_last_storage_path(active_dest_dir)

    # UI-Elemente für die Dauer des Downloads sperren
    btn_start_download.disabled = True
    btn_confirm_platform.disabled = True
    dropdown_platform.disabled = True
    text_model_search.disabled = True
    dropdown_model.disabled = True
    dropdown_gguf_file.disabled = True
    dropdown_target_disk.disabled = True
    progress_bar.layout.display = "block"
    progress_bar.value = 0
    label_status.value = "Initialisiere Speicherort..."
    
    with output_log:
        clear_output()
        print("=" * 60)
        print(f"--- START DOWNLOAD & INTEGRATION ({platform_choice.upper()}) ---")
        print(f"Priorisiertes Zielverzeichnis: {active_dest_dir}")
        print("=" * 60)
    
    try:
        print(f"[INFO] Erstelle Zielverzeichnis (falls nicht vorhanden): {active_dest_dir}")
        os.makedirs(active_dest_dir, exist_ok=True)

        if platform_choice == "ollama":
            raw_tag = text_custom_tag.value.strip() if selected_id == "CUSTOM_OLLAMA" else selected_id
            if not raw_tag:
                print("[FEHLER] Kein gültiger Ollama-Modelltag angegeben.")
                raise ValueError("Bitte einen gültigen Ollama-Modelltag angeben!")
    
            target_tag = normalize_model_tag(raw_tag)
            
            with output_log:
                print("[System] Beende blockierende Ollama-Prozesse...")
            print("[INFO] Suche nach laufenden Ollama-Prozessen zum Beenden...")
            
            killed_count = 0
            for proc in psutil.process_iter(['name']):
                try:
                    if proc.info['name'] and 'ollama' in proc.info['name'].lower():
                        proc.kill()
                        killed_count += 1
                except (psutil.NoSuchProcess, psutil.AccessDenied) as pe:
                    print(f"[DEBUG] Konnte Prozess nicht beenden (bereits beendet oder keine Rechte): {pe}")
            print(f"[✓] {killed_count} blockierende Ollama-Prozesse bereinigt.")
            time.sleep(2)
            
            with output_log:
                print(f"[System] Starte entkoppelten Ollama-Dienst auf Wunschpfad...")
            print(f"[INFO] Starte lokalen Ollama-Dienst mit OLLAMA_MODELS={active_dest_dir}...")
            
            env_copy = os.environ.copy()
            env_copy["OLLAMA_MODELS"] = str(Path(active_dest_dir).resolve())
            creationflags = subprocess.CREATE_NEW_PROCESS_GROUP if hasattr(subprocess, 'CREATE_NEW_PROCESS_GROUP') else 0
            subprocess.Popen(["ollama", "serve"], env=env_copy, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, creationflags=creationflags)
            time.sleep(4)
            print("[✓] Ollama-Dienst erfolgreich gestartet.")
            
            current_digest = None
            def ollama_callback(progress):
                nonlocal current_digest
                status = progress.get('status', '')
                completed = progress.get('completed', 0)
                total = progress.get('total', 0)
                digest = progress.get('digest', '')
                
                if digest != current_digest and digest:
                    with output_log:
                        print(f"[Layer {digest[:12]}] {status}")
                    current_digest = digest
                
                if total and completed and total > 0:
                    pct = min(int((completed / total) * 100), 100)
                    progress_bar.value = pct
                    label_status.value = f"[Pull] {status} ({completed // (1024**2)} / {total // (1024**2)} MB)"
                else:
                    label_status.value = f"[Pull] {status}"
            
            print(f"[INFO] Starte Ollama Pull für Modell: {target_tag}")
            redirected_client = ollama.Client(host="http://127.0.0.1:11434")
            for progress in redirected_client.pull(model=target_tag, stream=True):
                ollama_callback(progress)
            registered_name = target_tag
            allocated_size = 4.0
            print(f"[✓] Ollama Pull erfolgreich abgeschlossen für: {registered_name}")
            
        else: # Hugging Face
            if selected_id == "CUSTOM_HF":
                repo_id = text_custom_hf_repo.value.strip()
                raw_filename = text_custom_hf_file.value.strip()
            else:
                repo_id = selected_id
                raw_filename = dropdown_gguf_file.value
            
            print(f"[INFO] Hugging Face Download-Konfiguration - Repo: {repo_id!r}, Datei: {raw_filename!r}")
            if not repo_id or not raw_filename:
                print("[FEHLER] Repo-ID oder GGUF-Dateiname fehlt.")
                raise ValueError("Bitte Repo-ID und GGUF-Dateinamen angeben!")
            
            filename = os.path.basename(raw_filename)
            raw_tag = filename.lower().replace(".gguf", "").replace("_", "-").replace(".", "-")
            target_tag = normalize_model_tag(raw_tag)
            
            def hf_progress(msg):
                label_status.value = msg
                with output_log: print(f"[HF Hub] {msg}")
            def hf_build(st):
                label_status.value = f"[Ollama Build] {st}"
                with output_log: print(f"[Ollama Build] {st}")
            
            progress_bar.value = 30
            print(f"[INFO] Übergebe an HuggingFaceBackend für Download von '{filename}' aus '{repo_id}'...")
            local_path, allocated_size = hf_backend.download_and_create_model(
                repo_id=repo_id, filename=filename, target_model_tag=target_tag,
                dest_dir=active_dest_dir, progress_callback=hf_progress, build_callback=hf_build
            )
            registered_name = target_tag
            print(f"[✓] Hugging Face Modell erfolgreich heruntergeladen und registriert: {registered_name}")
        
        print(f"[INFO] Speichere aktive Modellkonfiguration...")
        ModelConfigManager.save_active_model(model_name=registered_name, size_gb=allocated_size, platform_name=platform_choice, config_dir=CONFIG_DIR)
        
        progress_bar.value = 100
        label_status.value = f"[✓] Erfolgreich! '{registered_name}' wurde auf Disk gesichert."
        print(f"[✓] Download-Prozess vollständig abgeschlossen für '{registered_name}'.")
        
    except Exception as err:
        print(f"[FEHLER] Kritischer Fehler im Download-Prozess: {err}")
        progress_bar.bar_style = "danger"
        label_status.value = f"[FEHLER] {err}"
        with output_log:
            print(f"\n[FEHLER] {err}")
        raise

def on_disk_selection_changed(change):
    """Wird aufgerufen, wenn sich die Laufwerksauswahl ändert, mit aktivem Logging."""
    print(f"[INFO] Laufwerksauswahl geändert. Alter Wert: {change.old!r} -> Neuer Wert: {change.new!r}")
    if change.new == "CUSTOM_PATH":
        print("[INFO] 'CUSTOM_PATH' ausgewählt. Blende Eingabefeld für benutzerdefinierten Pfad ein.")
        text_custom_storage_path.layout.display = "block"
    else:
        print("[INFO] Vordefiniertes Laufwerk ausgewählt. Blende benutzerdefiniertes Eingabefeld aus.")
        text_custom_storage_path.layout.display = "none"
        if change.new:
            print(f"[INFO] Speichere neuen Pfad automatisch ab: {change.new!r}")
            save_last_storage_path(change.new)

def on_custom_path_submitted(change):
    """Wird aufgerufen, wenn ein benutzerdefinierter Pfad eingegeben oder bestätigt wird, mit aktivem Logging."""
    print(f"[INFO] Benutzerdefinierter Pfad übermittelt. Neuer Wert: {change.new!r}")
    if change.new:
        cleaned_path = change.new.strip()
        print(f"[INFO] Speichere bereinigten benutzerdefinierten Pfad: {cleaned_path!r}")
        save_last_storage_path(cleaned_path)
    else:
        print("[INFO] Übermittelter benutzerdefinierter Pfad ist leer. Speicherung übersprungen.")

# UI CONTROLLER & EVENT LOGIC
CURRENT_LOADED_MODELS = []

def render_spec_card(model_item: dict) -> str:
    """Rendert die HTML-Spezifikationskarte für das ausgewählte Modell mit aktivem Logging."""
    print("[INFO] Starte Rendern der Modell-Spezifikationskarte...")    
    if not model_item:
        print("[HINWEIS] Kein Modell für die Spezifikationskarte übergeben. Rendere Leer-Zustand.")
        return "<div style='padding: 8px 10px; color: #6B7280; font-size: 12px;'>Kein Modell ausgewählt.</div>"
    model_id = model_item.get("id", "-")
    size = model_item.get("size", "-")
    category = model_item.get("category", "-")
    popularity = model_item.get("popularity", "-")
    platform_name = model_item.get("platform", "").upper()
    print(f"[DEBUG] Generiere Spec-Card für Modell: {model_id!r} (Plattform: {platform_name})")
    html = f"""
    <div style='background: #FFFFFF; border: 1px solid #E2E8F0; border-radius: 6px; padding: 10px; margin-top: 4px;'>
        <div style='display: flex; justify-content: space-between; align-items: center; margin-bottom: 6px;'>
            <span style='background-color: #3B82F6; color: white; font-weight: bold; padding: 2px 8px; border-radius: 4px; font-size: 11px;'>{platform_name}</span>
            <span style='font-weight: bold; font-size: 13px; color: #111827;'>{model_id}</span>
            <span style='background-color: #10B981; color: white; padding: 2px 8px; border-radius: 4px; font-size: 11px;'>RAM verfügbar: {TOTAL_RAM_GB} GB</span>
        </div>
        <div style='display: grid; grid-template-columns: repeat(3, 1fr); gap: 6px; background: #F9FAFB; padding: 6px; border-radius: 4px; font-size: 12px; color: #374151;'>
            <div><b>Größe:</b> {size}</div>
            <div><b>Schwerpunkt/Task:</b> {category}</div>
            <div><b>Downloads/Pulls:</b> {popularity}</div>
        </div>
    </div>
    """
    print("[✓] Spezifikationskarte erfolgreich gerendert.")
    return html


def on_search_query_changed(change):
    """Wird aufgerufen, wenn sich der Suchbegriff ändert, mit aktivem Logging."""
    query_val = change.new if change.new else ""
    print(f"[INFO] Suchfeld-Eingabe geändert. Neuer Suchbegriff: {query_val!r}")
    filter_model_options(query_val)

def on_platform_confirm_clicked(b):
    """Wird aufgerufen, wenn die Plattform-Auswahl bestätigt wird, mit aktivem Logging."""
    global CURRENT_LOADED_MODELS
    platform_type = dropdown_platform.value
    print(f"[INFO] Plattform-Bestätigung ausgelöst. Gewählte Plattform: {platform_type!r}")
    
    label_status.value = f"Lade Live-Modelle für '{platform_type}'..."
    btn_confirm_platform.disabled = True
    dropdown_model.disabled = True
    text_model_search.value = ""
    
    try:
        if platform_type == "ollama":
            print("[INFO] Rufe Modelle für Ollama ab (Limit: 80)...")
            CURRENT_LOADED_MODELS = ollama_backend.fetch_models(limit=80)
            dropdown_gguf_file.layout.display = "none"
            text_custom_tag.layout.display = "none"
            custom_hf_row.layout.display = "none"
            print("[✓] Ollama-spezifische UI-Elemente ausgeblendet.")
        else:
            print(f"[INFO] Rufe GGUF-Modelle für Hugging Face ab (Limit: 50)...")
            CURRENT_LOADED_MODELS = hf_backend.fetch_models(limit=50)
            dropdown_gguf_file.layout.display = "block"
            text_custom_tag.layout.display = "none"
            custom_hf_row.layout.display = "none"
            print("[✓] Hugging-Face-spezifische UI-Elemente angepasst (GGUF-Dropdown eingeblendet).")
        
        print(f"[INFO] {len(CURRENT_LOADED_MODELS)} Modelle geladen. Starte Aktualisierung der Filteroptionen...")
        filter_model_options("")
        label_status.value = f"{len(CURRENT_LOADED_MODELS)} Live-Modelle für '{platform_type.upper()}' bereit."
        print(f"[✓] Plattform-Wechsel erfolgreich abgeschlossen.")
        
    except Exception as err:
        print(f"[FEHLER] Fehler beim Laden der Live-Modelle für '{platform_type}': {err}")
        label_status.value = f"Fehler beim Laden: {err}"
    finally:
        btn_confirm_platform.disabled = False
        print("[DEBUG] Bestätigungs-Button für Plattformwechsel wieder aktiviert.")

def update_hf_file_dropdown(repo_id: str):
    """Aktualisiert die GGUF-Auswahl und bewertet jede Variante mit ✅ oder ❌ mit aktivem Logging."""
    print(f"[INFO] Starte Aktualisierung des GGUF-Dateidropdowns für Repository: {repo_id!r}")
    try:
        options = hf_backend.fetch_gguf_files_with_fit(repo_id)
        if options:
            print(f"[✓] {len(options)} GGUF-Optionen für Repo '{repo_id}' ermittelt. Aktualisiere Dropdown...")
            dropdown_gguf_file.options = options
            dropdown_gguf_file.disabled = False
            print("[✓] GGUF-Dropdown erfolgreich aktualisiert und aktiviert.")
        else:
            print(f"[WARNUNG] Keine .gguf-Dateien für Repository '{repo_id}' gefunden.")
            dropdown_gguf_file.options = [("Keine .gguf-Dateien gefunden", "")]
            dropdown_gguf_file.disabled = True
    except Exception as e:
        print(f"[FEHLER] Fehler beim Aktualisieren des GGUF-Dateidropdowns für '{repo_id}': {e}")
        dropdown_gguf_file.options = [("Fehler beim Laden der Dateien", "")]
        dropdown_gguf_file.disabled = True
        raise

def on_model_selection_changed(change):
    if not change.new:
        return
    selected_id = change.new
    
    if selected_id == "CUSTOM_OLLAMA":
        text_custom_tag.layout.display = "block"
        custom_hf_row.layout.display = "none"
        dropdown_gguf_file.layout.display = "none"
    elif selected_id == "CUSTOM_HF":
        text_custom_tag.layout.display = "none"
        custom_hf_row.layout.display = "flex"
        dropdown_gguf_file.layout.display = "none"
    else:
        text_custom_tag.layout.display = "none"
        custom_hf_row.layout.display = "none"
        if dropdown_platform.value == "huggingface":
            dropdown_gguf_file.layout.display = "block"
            update_hf_file_dropdown(selected_id)
        else:
            dropdown_gguf_file.layout.display = "none"
            
        matched = next((m for m in CURRENT_LOADED_MODELS if m['id'] == selected_id), None)
        if matched:
            html_spec_card.value = render_spec_card(matched)

# DOWNLOAD EXECUTION
def on_model_selection_changed(change):
    """Wird aufgerufen, wenn sich die Modell-Auswahl ändert, mit aktivem Logging."""
    if not change.new:
        print("[INFO] Modell-Auswahl geändert, aber kein neuer Wert vorhanden (leer).")
        return
    
    selected_id = change.new
    print(f"[INFO] Modell-Auswahl geändert. Neuer Wert / Ausgewählte ID: {selected_id!r}")
    
    if selected_id == "CUSTOM_OLLAMA":
        print("[INFO] 'CUSTOM_OLLAMA' ausgewählt. Blende Ollama-Tag-Eingabefeld ein.")
        text_custom_tag.layout.display = "block"
        custom_hf_row.layout.display = "none"
        dropdown_gguf_file.layout.display = "none"
    elif selected_id == "CUSTOM_HF":
        print("[INFO] 'CUSTOM_HF' ausgewählt. Blende Hugging-Face-Eingabezeile ein.")
        text_custom_tag.layout.display = "none"
        custom_hf_row.layout.display = "flex"
        dropdown_gguf_file.layout.display = "none"
    else:
        print("[INFO] Reguläres Modell ausgewählt. Setze Layout-Elemente zurück...")
        text_custom_tag.layout.display = "none"
        custom_hf_row.layout.display = "none"
        
        if dropdown_platform.value == "huggingface":
            print("[INFO] Aktive Plattform ist Hugging Face. Blende GGUF-Dropdown ein und lade Dateiliste...")
            dropdown_gguf_file.layout.display = "block"
            update_hf_file_dropdown(selected_id)
        else:
            print("[INFO] Aktive Plattform ist Ollama. GGUF-Dropdown bleibt ausgeblendet.")
            dropdown_gguf_file.layout.display = "none"
            
        matched = next((m for m in CURRENT_LOADED_MODELS if m['id'] == selected_id), None)
        if matched:
            print(f"[✓] Metadaten für Modell '{selected_id}' in geladenen Modellen gefunden. Aktualisiere Spezifikationskarte...")
            html_spec_card.value = render_spec_card(matched)
        else:
            print(f"[WARNUNG] Keine passenden Metadaten für Modell-ID '{selected_id}' in 'CURRENT_LOADED_MODELS' gefunden.")




# --- GLOBALE LAYOUT- & WIDGET-HELPER ---
_DEFAULT_TEXT_LAYOUT = widgets.Layout(width=FULL_WIDTH, max_width=MAX_WIDTH, margin="0 0 8px 0")
_DEFAULT_STYLE = {"description_width": LABEL_WIDTH}

def _create_text(placeholder: str, description: str, display: str = "block") -> widgets.Text:
    """Erstellt standardisierte Text-Widgets mit einheitlichem Layout."""
    return widgets.Text(
        placeholder=placeholder,
        description=description,
        style=_DEFAULT_STYLE,
        layout=widgets.Layout(width=FULL_WIDTH, max_width=MAX_WIDTH, margin="0 0 8px 0", display=display)
    )

def _create_dropdown(description: str, options=None, disabled=False, display="block") -> widgets.Dropdown:
    """Erstellt standardisierte Dropdown-Widgets."""
    return widgets.Dropdown(
        options=options or [],
        description=description,
        style=_DEFAULT_STYLE,
        layout=widgets.Layout(width=FULL_WIDTH, max_width=MAX_WIDTH, margin="0 0 8px 0", display=display),
        disabled=disabled
    )

def _create_section(title: str, subtitle: str, children: list) -> widgets.VBox:
    """Erstellt einen standardisierten Sektionen-Container im einheitlichen UI-Design."""
    return widgets.VBox(
        [
            widgets.HTML(f"<div style='font-size: 13px; font-weight: 700; color: #1E293B; margin-bottom: 2px;'>{title}</div>"),
            widgets.HTML(f"<div style='font-size: 11px; color: #64748B; margin-bottom: 6px;'>{subtitle}</div>"),
            *children
        ],
        layout=widgets.Layout(
            width=FULL_WIDTH, max_width=MAX_WIDTH, padding="12px 14px", margin="0 0 10px 0",
            border="1px solid #E2E8F0", border_radius="8px", background_color="#F8FAFC"
        )
    )

# --- SEKTION 1 WIDGETS (PLATTFORM-AUSWAHL) ---
dropdown_platform = widgets.Dropdown(
    options=[
        ("🦙 Ollama (Offizielle Library & Live-Registry)", "ollama"),
        ("🤗 Hugging Face (GGUF Community Modelle - Top Downloads)", "huggingface")
    ],
    value="ollama",
    description="Plattform:",
    style=_DEFAULT_STYLE,
    layout=widgets.Layout(flex="1 1 auto", min_width="300px", max_width="480px")
)

btn_confirm_platform = widgets.Button(
    description="OK - Plattform bestätigen",
    button_style="info",
    icon="check",
    layout=widgets.Layout(width="210px", height="36px")
)

platform_row = widgets.HBox(
    [dropdown_platform, btn_confirm_platform],
    layout=widgets.Layout(width=FULL_WIDTH, align_items="center", justify_content="flex-start", gap="12px", margin="6px 0 0 0")
)

section_platform = _create_section(
    "📍 Sektion 1: Plattform & Registry auswählen",
    "Wähle die Quelle aus und lade die Live-Modelle mit dem Bestätigungs-Button.",
    [platform_row]
)

# --- SEKTION 2 WIDGETS (SUCHE & MODELL-AUSWAHL) ---
text_model_search = _create_text("Suchbegriff eingeben (z.B. coder, vision, 8b, qwen, llama, math...)", "Modell suchen:")
dropdown_model = _create_dropdown("Modell wählen:", disabled=True)
dropdown_gguf_file = _create_dropdown("GGUF-Datei:", display="none")
text_custom_tag = _create_text("z.B. deepseek-coder:6.7b oder mistral-nemo", "Ollama Tag:", display="none")

text_custom_hf_repo = widgets.Text(placeholder="z.B. TheBloke/Mistral-7B-Instruct-v0.2-GGUF", description="HF Repo ID:", style=_DEFAULT_STYLE, layout=widgets.Layout(flex="1 1 auto", min_width="260px"))
text_custom_hf_file = widgets.Text(placeholder="z.B. mistral-7b-instruct-v0.2.Q4_K_M.gguf", description="HF Dateiname:", style=_DEFAULT_STYLE, layout=widgets.Layout(flex="1 1 auto", min_width="260px"))

custom_hf_row = widgets.HBox(
    [text_custom_hf_repo, text_custom_hf_file],
    layout=widgets.Layout(width=FULL_WIDTH, max_width=MAX_WIDTH, gap="10px", margin="0 0 8px 0", display="none")
)

html_spec_card = widgets.HTML(
    value="<div style='padding: 10px; border-left: 4px solid #3B82F6; background-color: #F1F5F9; border-radius: 6px; font-size: 12px; color: #475569;'>Wähle eine Plattform in Sektion 1, um Live-Modelle anzuzeigen.</div>",
    layout=widgets.Layout(width=FULL_WIDTH, max_width=MAX_WIDTH, margin="4px 0 0 0")
)

section_model = _create_section(
    "🔍 Sektion 2: Echtzeit-Suche & Modell-Auswahl",
    "Filtere die Live-Modelle nach Stichworten und wähle die gewünschte Variante.",
    [text_model_search, dropdown_model, dropdown_gguf_file, text_custom_tag, custom_hf_row, html_spec_card]
)

# --- SEKTION 3 WIDGETS (SPEICHERORT, DOWNLOAD & INTEGRATION) ---
dropdown_target_disk = _create_dropdown("Ziel-Laufwerk:", options=get_system_drives())
text_custom_storage_path = _create_text("z.B. D:\\AI_Models oder /mnt/ext_drive/models", "Pfad (Custom):", display="none")

# Gespeicherten Pfad beim Start laden und vorauswählen
saved_path = load_last_storage_path()
if saved_path:
    matching_option = next((opt[1] for opt in dropdown_target_disk.options if opt[1] == saved_path), None)
    if matching_option:
        dropdown_target_disk.value = matching_option
    else:
        dropdown_target_disk.value = "CUSTOM_PATH"
        text_custom_storage_path.value = saved_path
        text_custom_storage_path.layout.display = "block"

btn_start_download = widgets.Button(
    description="OK - Download & In Ollama einbinden",
    button_style="success",
    icon="cloud-download",
    layout=widgets.Layout(width="300px", height="38px", margin="0 0 8px 0"),
    disabled=True
)

progress_bar = widgets.IntProgress(
    value=0, min=0, max=100, description="Fortschritt:", bar_style="info",
    style={"description_width": LABEL_WIDTH, "bar_color": "#10B981"},
    layout=widgets.Layout(width=FULL_WIDTH, max_width=MAX_WIDTH, margin="0 0 4px 0", display="none")
)

label_status = widgets.Label(value="Warte auf Konfiguration...", layout=widgets.Layout(margin="0 0 6px 0"))

output_log = widgets.Output(
    layout=widgets.Layout(
        width=FULL_WIDTH, max_width=MAX_WIDTH, height="150px",
        border="1px solid #CBD5E1", border_radius="6px", padding="8px",
        overflow="auto", background_color="#0F172A"
    )
)

section_download = _create_section(
    "🚀 Sektion 3: Speicherort, Download & Einbindung",
    "Wähle das Ziellaufwerk, starte den Download und registriere das Modell in Ollama.",
    [dropdown_target_disk, text_custom_storage_path, btn_start_download, progress_bar, label_status,
     widgets.HTML("<div style='font-size: 11px; font-weight: 700; color: #475569; margin: 4px 0;'>Live-Prozessprotokoll:</div>"), output_log]
)

# Überarbeitung des Master-Containers (Hintergrundfarbe von Sektion 3 angepasst, um einheitlich zu bleiben)
section_download.layout.background_color = "#FFFFFF"

# MASTER DASHBOARD CONTAINER
master_dashboard = widgets.VBox([
    widgets.HTML(
        "<div style='display: flex; align-items: center; justify-content: space-between; border-bottom: 2px solid #E2E8F0; padding-bottom: 8px; margin-bottom: 12px;'>"
        "<h3 style='margin: 0; color: #0F172A; font-family: sans-serif; font-size: 16px;'>🎛️ Manuelles Modell-Management & Download-Utility</h3>"
        "<span style='background-color: #3B82F6; color: white; padding: 2px 8px; border-radius: 10px; font-size: 11px; font-weight: bold;'>Mai_AI Local</span>"
        "</div>"
    ),
    section_platform,
    section_model,
    section_download
], layout=widgets.Layout(
    width=FULL_WIDTH, max_width="800px", padding="16px", margin="10px auto",
    border="1px solid #CBD5E1", border_radius="10px", background_color="#FFFFFF"
))

dropdown_target_disk.observe(on_disk_selection_changed, names='value')
text_custom_storage_path.observe(on_custom_path_submitted, names='value')
text_model_search.observe(on_search_query_changed, names='value')
btn_confirm_platform.on_click(on_platform_confirm_clicked)
dropdown_model.observe(on_model_selection_changed, names='value')
btn_start_download.on_click(on_start_download_clicked)

# Erste Initialisierung der Plattform
on_platform_confirm_clicked(None)
display(master_dashboard)

In [ ]:
from ollama import Client
from pathlib import Path
import os
import re
import requests
import json
import subprocess

# Definiere den Standard-Ablagepfad (kann dynamisch übergeben werden)
HF_MODELS_DIR = r"C:\Users\MacBookAir\Desktop\GitHub\Offline_AI\data\models\huggingface"

def clean_model_name(filename_stem: str) -> str:
    """Extrahiert Modell-Parameter und erstellt einen sauberen Ollama-Tag."""
    print(f"       [DEBUG-EXTRACT] Starte Analyse für Dateistem: '{filename_stem}'")
    name_lower = filename_stem.lower()
    
    model_patterns = ["deepseek", "llama", "codestral", "mistral", "qwen", "phi", "gemma"]
    detected_model = "model"
    for m in model_patterns:
        if m in name_lower:
            detected_model = m
            break
            
    version_match = re.search(r'(v\d+|\d+\.\d+)', name_lower)
    detected_version = version_match.group(1) if version_match else ""
    
    intensity_keywords = ["flash", "pro", "lite", "support", "dspark", "chat", "instruct", "base"]
    found_intensities = [kw for kw in intensity_keywords if kw in name_lower]
    detected_intensity = "-".join(found_intensities[:2])
    
    parts = [detected_model]
    if detected_version: parts.append(detected_version)
    if detected_intensity: parts.append(detected_intensity)
        
    clean_name = re.sub(r'[^a-z0-9._-]', '-', "-".join(parts))
    clean_name = re.sub(r'-+', '-', clean_name).strip('-_.')
    
    if len(clean_name) > 35: clean_name = clean_name[:35].strip('-_.')
    final_tag = f"{clean_name}:latest"
    print(f"       [DEBUG-EXTRACT] -> Finaler Übergabe-Tag: '{final_tag}'")
    return final_tag

def verify_model_integrity(client: Client, model_tag: str) -> bool:
    """
    Führt einen echten Test-Call durch, um zu prüfen, ob Ollama 
    die Architektur des Modells laden kann. Gibt False bei Inkompatibilität zurück.
    """
    print(f"       -> [INTEGRITY-CHECK] Teste Ladefähigkeit für '{model_tag}'...")
    try:
        client.generate(model=model_tag, prompt="Test", options={"num_predict": 1})
        print(f"       -> [INTEGRITY-CHECK] [✓] Modell '{model_tag}' ist voll funktionsfähig.")
        return True
    except Exception as err:
        err_str = str(err)
        if "unknown model architecture" in err_str or "500 Internal Server Error" in err_str:
            print(f"       -> [INTEGRITY-CHECK] [X] ARCHITEKTUR-FEHLER ERKANNT: Ollama unterstützt die Struktur dieses Modells noch nicht.")
            print(f"         Details: {err_str.strip()}")
        else:
            print(f"       -> [INTEGRITY-CHECK] [!] Warnung bei Test-Inferenz: {err_str}")
        return False

def sync_local_gguf_files_to_ollama(models_dir: str, ollama_host: str = "http://127.0.0.1:11434"):
    print("\n[STEP 1] Initializing path resolution...")
    models_dir = Path(models_dir).resolve()
    
    if not models_dir.exists():
        print(f"[FEHLER] Der angegebene Ablagepfad existiert nicht: {models_dir}")
        return

    print(f"\n[STEP 2] Scanning for .gguf files in: {models_dir}")
    gguf_files = list(models_dir.glob("*.gguf"))
    print(f" -> Gefunden: {len(gguf_files)} GGUF-Dateien.")
    
    print("\n[STEP 3] Connecting to Ollama API client...")
    ollama_client = Client(host=ollama_host)
    try:
        existing_models = {str(m.model).strip().lower() for m in ollama_client.list().models}
    except Exception as e:
        print(f"[FEHLER] Konnte keine Verbindung zu Ollama herstellen ({ollama_host}): {e}")
        return
    
    print("-" * 60)
    print("[STEP 4] Processing files and executing synchronization...")
    
    registered_count, skipped_count, error_count = 0, 0, 0
    
    for idx, gguf_file in enumerate(gguf_files, 1):
        print(f"\n  [{idx}/{len(gguf_files)}] Evaluating: {gguf_file.name}")
        model_name = clean_model_name(gguf_file.stem)
        
        if model_name in existing_models:
            print("       -> Status: [SKIP] Modell existiert bereits.")
            skipped_count += 1
            continue
        
        modelfile_path = gguf_file.parent / f"Modelfile_{gguf_file.stem}"
        absolute_gguf_path = str(gguf_file.absolute()).replace(chr(92), '/')
        
        with open(modelfile_path, "w", encoding="utf-8") as f:
            f.write(f"FROM {absolute_gguf_path}\n")
        
        # API Create Request
        url = f"{ollama_host}/api/create"
        payload = {"model": model_name, "modelfile": f"FROM {absolute_gguf_path}\n", "stream": True}
        response = requests.post(url, json=payload, stream=True)
        
        success = False
        for line in response.iter_lines():
            if line:
                data = json.loads(line.decode('utf-8'))
                if "error" in data:
                    print(f"         [Ollama ERROR] {data['error']}")
                    if "neither 'from' or 'files'" in data['error'] or "from" in data['error'].lower():
                        print("       -> [FALLBACK] Nutze CLI...")
                        if subprocess.run(["ollama", "create", model_name, "-f", str(modelfile_path.absolute())]).returncode == 0:
                            success = True
                if data.get("status") == "success" or data.get("done"): 
                    success = True
        
        if success: 
            registered_count += 1
        else: 
            error_count += 1
            
        if modelfile_path.exists(): 
            modelfile_path.unlink()

    # --- SCHRITT 5: SYSTEMATISCHER TEST ALLER VORHANDENEN MODELLE ---
    print("\n" + "=" * 60)
    print("=== [STEP 5] VOLLSTÄNDIGER INTEGRITÄTS-CHECK ALLER LOKALEN MODELLE ===")
    
    local_models = ollama_client.list()
    CONFIG_DIR = "./config"
    os.makedirs(CONFIG_DIR, exist_ok=True)
    active_cfg_file = os.path.join(CONFIG_DIR, "active_model_config.json")
    active_model_tag = "llama3.1:latest"
    
    if os.path.exists(active_cfg_file):
        with open(active_cfg_file, "r", encoding="utf-8") as f:
            active_model_tag = json.load(f).get("model_name", active_model_tag)

    healthy_models, broken_models = 0, 0

    for i, item in enumerate(local_models.models, 1):
        size_mb = item.size // (1024**2) if item.size else 0
        active_flag_str = " (AKTIV)" if item.model.startswith(active_model_tag.split(":")[0]) else ""
        
        print(f"\n--- Prüfe Modell [{i}/{len(local_models.models)}]: {item.model} ({size_mb} MB){active_flag_str} ---")
        
        if verify_model_integrity(ollama_client, item.model):
            healthy_models += 1
        else:
            broken_models += 1

    # --- ZUSAMMENFASSUNG & STATUS ---
    print("\n" + "=" * 60)
    print("=== LOKALE OLLAMA-MODELLE & STATUS ===")
    print(f"   Gefundene GGUF-Dateien: {len(gguf_files)}")
    print(f"   Neu registriert: {registered_count} | Übersprungen: {skipped_count}")
    print(f"   Modelle insgesamt in Ollama: {len(local_models.models)}")
    print(f"   [✓] Funktionsfähig: {healthy_models} | [X] Fehlerhaft/Inkompatibel: {broken_models}")
    print("=" * 60)
    
    for i, item in enumerate(local_models.models, 1):
        size_mb = item.size // (1024**2) if item.size else 0
        active_flag_str = " (AKTIV)" if item.model.startswith(active_model_tag.split(":")[0]) else ""
        print(f" {i}. {item.model:<35} | {size_mb:>6} MB{active_flag_str}")
    print("=" * 60)

    # Test-Inferenz mit dem aktiven Standardmodell als Abschluss
    prompt = "Erkläre kurz in einem Satz den Vorteil lokaler KI-Modelle."
    print(f"\n[TEST-INFERENZ] Aktives Modell: '{active_model_tag}'")
    try:
        res = ollama_client.generate(model=active_model_tag, prompt=prompt)
        print("--- ANTWORT ---")
        print(res.get('response', '').strip())
        print("----------------\n[✓] Offline-Inferenz erfolgreich!")
    except Exception as err:
        print(f"[!] Test fehlgeschlagen: {err}")

# Ausführung
if __name__ == "__main__":
    sync_local_gguf_files_to_ollama(HF_MODELS_DIR)

In [ ]:
!pip uninstall -y llama-cpp-python
from pathlib import Path
from llama_cpp import Llama

class DeepSeekAgent:
    def __init__(self, model_path: str, n_ctx: int = 4096, n_gpu_layers: int = -1):
        """
        Initialisiert den Agenten direkt mit der GGUF-Modelldatei,
        unabhängig von Ollama oder eventuellen Architektur-Einschränkungen.
        """
        self.model_path = Path(model_path)
        
        if not self.model_path.exists():
            raise FileNotFoundError(f"Modell-Datei nicht gefunden unter: {self.model_path}")
            
        print(f"[AGENT-INIT] Lade DeepSeek-Modell direkt: {self.model_path.name}...")
        print("[AGENT-INIT] Dies kann je nach Modellgröße einen Moment dauern...")
        
        # Direktes Laden der Gewichte in den Speicher / die GPU
        self.llm = Llama(
            model_path=str(self.model_path),
            n_ctx=n_ctx,
            n_gpu_layers=n_gpu_layers, # -1 nutzt alle verfügbaren GPU-Layer (falls CUDA aktiv ist)
            verbose=False
        )
        print("[AGENT-INIT] [✓] Agent ist einsatzbereit!")

    def query(self, prompt: str, max_tokens: int = 512, temperature: float = 0.7) -> str:
        """
        Sendet eine Anfrage an den Agenten und gibt die generierte Antwort zurück.
        """
        print(f"\n[AGENT-QUERY] Verarbeite Anfrage...")
        
        output = self.llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            stop=["<|endoftext|>", "</s>"],
            echo=False
        )
        
        return output["choices"][0]["text"].strip()

# --- Ausführung und Test des Agenten ---
if __name__ == "__main__":
    # Pfad zu deiner DeepSeek GGUF-Datei
    MODEL_FILE = r"C:\Users\MacBookAir\Desktop\GitHub\Offline_AI\data\models\huggingface\DeepSeek-V4-Flash-DSpark-support-0731.gguf"
    
    try:
        # Agenten instanziieren
        agent = DeepSeekAgent(model_path=MODEL_FILE)
        
        # Hier kannst du deinen beliebigen Text / Query eintragen
        user_query = "Erkläre mir in zwei Sätzen, wie eine lokale KI-Infrastruktur die Datensicherheit erhöht."
        
        print(f"Eingeagte Query: '{user_query}'")
        
        # Antwort generieren und ausgeben
        response = agent.query(prompt=user_query)
        
        print("\n--- AGENTEN-ANTWORT ---")
        print(response)
        print("------------------------")
        
    except Exception as e:
        print(f"[!] Fehler beim Ausführen des Agenten: {e}")

### 🧪 Schritt 4: Modell-Inspektion & Offline-Test
Auflistung der lokal registrierten Ollama-Modelle und Ausführung eines Test-Prompts zur Verifikation.

In [ ]:
# Lokal registrierte Modelle abrufen
local_models = service_manager.client.list()

# Aktive Konfiguration ermitteln
active_cfg_file = os.path.join(CONFIG_DIR, "active_model_config.json")
active_model_tag = "llama3.1:latest"
if os.path.exists(active_cfg_file):
    with open(active_cfg_file, "r", encoding="utf-8") as f:
        cfg = json.load(f)
        active_model_tag = cfg.get("model_name", active_model_tag)

print("=" * 60)
print("=== LOKALE OLLAMA-MODELLE ===")
for i, item in enumerate(local_models.models, 1):
    size_mb = item.size // (1024**2) if item.size else 0
    active_flag = " (AKTIV)" if item.model.startswith(active_model_tag.split(":")[0]) else ""
    print(f" {i}. {item.model:<35} | {size_mb:>6} MB{active_flag}")
print("=" * 60)

# Test-Inferenz
prompt = "Erkläre kurz in einem Satz den Vorteil lokaler KI-Modelle."
print(f"\n[TEST-INFERENZ] Modell: '{active_model_tag}'")
print(f"Prompt: '{prompt}'\n")

try:
    res = service_manager.client.generate(model=active_model_tag, prompt=prompt)
    print("--- ANTWORT ---")
    print(res.get('response', '').strip())
    print("----------------")
    print("[✓] Offline-Inferenz erfolgreich!")
except Exception as err:
    print(f"[Hinweis] Test nicht ausgeführt: {err}")

---
### 🔄 Nächste Schritte
Das Modell ist nun offline in Ollama bereitgestellt und in `config/active_model_config.json` hinterlegt.

* 👉 **[02_dockereinstellung.ipynb](file:notebooks/02_dockereinstellung.ipynb)** (Traefik-Gateway & Multi-User Isolation)
* 👉 **[03_html_embed.ipynb](file:notebooks/03_html_embed.ipynb)** (Streamlit Web-UI & HTML Embedding)
* 👉 **[Srart_mai_ai.ipynb](file:notebooks/Srart_mai_ai.ipynb)** (Hauptsystem-Start)